# BandDPER — Training Notebook

**BandDPER** *(Band-Aware Dual-Perspective Residual Network)* is a neural evaluation function for the board game **Escampe**.

This notebook:
1. Installs all dependencies
2. Loads the Escampe dataset from the Hugging Face Hub (`Bluefir/escampe-dataset`)
3. Trains BandDPER using AdamW + Cosine Annealing + MSE loss
4. Exports weights to `.pth` and folded JSON
5. Uploads the trained model to the Hugging Face Hub (`Bluefir/BandDPER`)

**Runtime:** GPU recommended (Colab → Runtime → Change runtime type → T4 GPU).

## 1. Install Dependencies

In [1]:
import os
import sys
import shutil

# Reset to /content to fix 'cannot access parent directories' error
%cd /content

# Clean up any partial/broken clones
if os.path.exists("app5-ai-game-project"):
    shutil.rmtree("app5-ai-game-project")

# Clone fresh
print("Cloning repository...")
!git clone https://github.com/Wubpooz/app5-ai-game-project.git

# Setup paths
repo_root = os.path.abspath("app5-ai-game-project")
repo_src = os.path.join(repo_root, "escampe/src")

print(f"Adding to sys.path: {repo_src}")
if repo_src not in sys.path:
    sys.path.append(repo_src)

# Install missing dependencies
%pip install -q torch torchvision datasets huggingface_hub numpy

/content
Cloning repository...
Cloning into 'app5-ai-game-project'...
remote: Enumerating objects: 1039, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 1039 (delta 6), reused 7 (delta 5), pack-reused 1024 (from 1)
Receiving objects: 100% (1039/1039), 10.49 MiB | 15.99 MiB/s, done.
Resolving deltas: 100% (507/507), done.
Adding to sys.path: /content/app5-ai-game-project/escampe/src


## 2. Imports & Device Setup

In [2]:
# Move into the model directory for local imports to work reliably
%cd /content/app5-ai-game-project/escampe/src/model

# Verify contents
!ls

/content/app5-ai-game-project/escampe/src/model
dataset.py  __init__.py  README.md	   train_colab.ipynb
export.py   model.py	 requirements.txt  train.py


In [ ]:
import json
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from datasets import load_dataset
from huggingface_hub import HfApi

from model import BandDPER
from dataset import BAND, BAND_MASK, build_tensor, EscampeDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Configuration

In [ ]:
# == Dataset ==================================================================
HF_DATASET_REPO   = "Bluefir/escampe-dataset"   # source dataset on the Hub
DATASET_CONFIG    = "all"                        # config to load ("all", "dataset_50k_d5", etc.)

# == Model upload =============================================================
HF_MODEL_REPO     = "Bluefir/BandDPER"           # destination model repo on the Hub

# == Training hyperparameters =================================================
EPOCHS            = 40
BATCH_SIZE        = 256
LR                = 1e-3
WEIGHT_DECAY      = 5e-4
VAL_SPLIT         = 0.1
NUM_RES_BLOCKS    = 3

# == Output files =============================================================
CHECKPOINT_PTH    = "banddper.pth"               # best checkpoint (full state dict)
WEIGHTS_JSON      = "banddper_weights.json"      # BN-folded weights for Java inference

print("Configuration loaded.")

## 4. Board Constants & Feature Engineering

These are ported directly from `dataset.py`. They encode the Escampe board topology (the *liseret* band map) and implement:
- Broken-path move generation via DFS
- Unicorn escape counting
- Forced-pass detection
- The 16-channel `[16, 6, 6]` board tensor constructor

## 5. Dataset — Load from Hugging Face

We load the raw JSON records from `Bluefir/escampe-dataset` and dynamically build the 16-channel `[16, 6, 6]` perspective tensors and scalar signals on the fly.

In [ ]:
class EscampeHFDataset(Dataset):
    """PyTorch Dataset backed by a Hugging Face `datasets` split.

    Each raw record contains:
      white_paladins, white_unicorn, black_paladins, black_unicorn  — bitboard integers
      required_band  — int in {0,1,2,3}
      white_to_move  — bool
      score          — float in [-1, +1]
    """
    def __init__(self, hf_split):
        print(f"Building tensors for {len(hf_split):,} records…")
        self.data = []
        for entry in hf_split:
            white_to_move = bool(entry['white_to_move'])
            score         = float(entry['score'])

            me_persp  = 'white' if white_to_move else 'black'
            opp_persp = 'black' if white_to_move else 'white'

            x_me,  esc_me,  esc_opp_me,  fp_me  = build_tensor(entry, me_persp)
            x_opp, esc_opp, esc_me_opp, fp_opp      = build_tensor(entry, opp_persp)

            self.data.append((
                torch.tensor(x_me, dtype=torch.float32),
                torch.tensor(x_opp, dtype=torch.float32),
                torch.tensor(esc_me, dtype=torch.float32),
                torch.tensor(esc_opp_me, dtype=torch.float32),
                torch.tensor(fp_me, dtype=torch.float32),
                torch.tensor(score, dtype=torch.float32),
            ))

            # Data augmentation
            self.data.append((
                torch.tensor(x_opp,  dtype=torch.float32),
                torch.tensor(x_opp,   dtype=torch.float32),
                torch.tensor(esc_opp, dtype=torch.float32),
                torch.tensor(esc_me_opp,  dtype=torch.float32),
                torch.tensor(fp_opp,     dtype=torch.float32),
                torch.tensor(-score, dtype=torch.float32),   # flipped score
            ))

        print(f"Done — {len(self.data):,} examples ready.")

    def __len__(self):        return len(self.data)
    def __getitem__(self, i): return self.data[i]


# == Load from Hub =============================================================
print(f"Loading '{DATASET_CONFIG}' split from {HF_DATASET_REPO}…")
hf_raw = load_dataset(HF_DATASET_REPO, DATASET_CONFIG, split="train")
print(f"Raw records: {len(hf_raw):,}")

dataset = EscampeHFDataset(hf_raw)

## 6. Train / Val Split & DataLoaders

In [ ]:
val_size   = int(len(dataset) * VAL_SPLIT)
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
print(f"Train: {train_size:,}  |  Val: {val_size:,}")

# Use 2 workers in Colab (avoids CUDA-multiprocessing issues)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE == "cuda"), drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE == "cuda"), drop_last=False)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

## 7. Model Definition

### Architecture overview

| Stage | Detail |
|-------|--------|
| **Shared Spatial Encoder** | `Conv2d(16→32, 3×3) → BN → ReLU → Conv2d(32→64, 3×3, dil=2) → BN → ReLU → Flatten → Linear(2304→128) → BN → ClippedReLU[0,1]` |
| **Siamese fusion** | Encode `x_me` and `x_opp` independently with *shared* weights → concat → `[B, 256]` |
| **Scalar injection** | Append `escape_me`, `escape_opp` → `[B, 258]` |
| **Residual trunk** | 3 × `ResBlock(258)` with ClippedReLU |
| **Output head** | `Linear(258→64) → ReLU → Linear(64→1) + w_forced_pass * forced_pass → tanh` |

In [ ]:
# The BandDPER model definition is imported directly from `model.model`.
# We instantiate the model using the configuration:
model = BandDPER(num_res_blocks=NUM_RES_BLOCKS).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"BandDPER — total params: {total_params:,}  |  trainable: {trainable:,}")
print(f"Approx. size (float32): {total_params * 4 / 1e6:.2f} MB")

## 8. Training Loop

**Hyperparameters recap:**

| Hyperparameter | Value |
|----------------|-------|
| Optimiser | AdamW |
| Learning rate | 1e-3 |
| Weight decay | 1e-4 |
| LR schedule | CosineAnnealingWarmRestarts (T₀ = 40) |
| Gradient clipping | 1.0 |
| Batch size | 256 |
| Loss | MSE |
| `drop_last` | True (avoids BatchNorm1d crash on batch of 1) |

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1)
criterion = nn.MSELoss()

best_val   = float('inf')
train_hist = []
val_hist   = []

for epoch in range(1, EPOCHS + 1):
    # == Train ==================================================================
    model.train()
    train_loss = 0.0
    for x_me, x_opp, esc_me, esc_opp, fp, targets in train_loader:
        x_me    = x_me.to(DEVICE, non_blocking=True)
        x_opp   = x_opp.to(DEVICE, non_blocking=True)
        esc_me  = esc_me.to(DEVICE, non_blocking=True)
        esc_opp = esc_opp.to(DEVICE, non_blocking=True)
        fp      = fp.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        preds = model(x_me, x_opp, esc_me, esc_opp, fp)
        loss  = criterion(preds, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(targets)
    train_loss /= train_size

    # == Validate ===============================================================
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_me, x_opp, esc_me, esc_opp, fp, targets in val_loader:
            x_me    = x_me.to(DEVICE, non_blocking=True)
            x_opp   = x_opp.to(DEVICE, non_blocking=True)
            esc_me  = esc_me.to(DEVICE, non_blocking=True)
            esc_opp = esc_opp.to(DEVICE, non_blocking=True)
            fp      = fp.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            preds   = model(x_me, x_opp, esc_me, esc_opp, fp)
            val_loss += criterion(preds, targets).item() * len(targets)
    val_loss /= val_size
    scheduler.step()

    train_hist.append(train_loss)
    val_hist.append(val_loss)

    tag = ""
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PTH)
        tag = f"  ✓ new best"

    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"train={train_loss:.5f} | val={val_loss:.5f} | "
          f"lr={scheduler.get_last_lr()[0]:.2e}{tag}")

print(f"\nTraining complete. Best val loss: {best_val:.5f} → {CHECKPOINT_PTH}")

## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, EPOCHS + 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs_range, train_hist, label="Train MSE", linewidth=2)
ax.plot(epochs_range, val_hist,   label="Val MSE",   linewidth=2, linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("BandDPER — Training Curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
print("Saved: training_curves.png")

## 10. Load Best Checkpoint

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PTH, map_location=DEVICE, weights_only=True))
model.eval()
print(f"Best checkpoint loaded from '{CHECKPOINT_PTH}'.")

# == Quick sanity check on a single batch =====================================
with torch.no_grad():
    x_me, x_opp, esc_me, esc_opp, fp, targets = next(iter(val_loader))
    preds = model(
        x_me.to(DEVICE, non_blocking=True), x_opp.to(DEVICE, non_blocking=True),
        esc_me.to(DEVICE, non_blocking=True), esc_opp.to(DEVICE, non_blocking=True), fp.to(DEVICE, non_blocking=True)
    ).cpu()

print(f"Prediction range  : [{preds.min():.3f}, {preds.max():.3f}]")
print(f"Target range      : [{targets.min():.3f}, {targets.max():.3f}]")
mse  = F.mse_loss(preds, targets).item()
mae  = (preds - targets).abs().mean().item()
print(f"Batch MSE: {mse:.5f}  |  Batch MAE: {mae:.5f}")

## 11. Export — BatchNorm-Folded Weights (JSON)

BatchNorm parameters are folded into the preceding conv/linear weights so Java inference requires no BN operations at runtime.

In [ ]:
from export import fold_batchnorm_conv, fold_batchnorm_linear

enc = model.encoder

# Encoder — fold BN layers
w_conv1, b_conv1 = fold_batchnorm_conv(enc.conv1, enc.bn1)
w_conv2, b_conv2 = fold_batchnorm_conv(enc.conv2, enc.bn2)
w_proj,  b_proj  = fold_batchnorm_linear(enc.proj, enc.bn3)

# ResBlock linears — fold BN layers
folded_blocks = []
for block in model.trunk:
    w_l1, b_l1 = fold_batchnorm_linear(block.l1, block.bn1)
    w_l2, b_l2 = fold_batchnorm_linear(block.l2, block.bn2)
    folded_blocks.append({
        "l1_weight": w_l1.tolist(), "l1_bias": b_l1.tolist(),
        "l2_weight": w_l2.tolist(), "l2_bias": b_l2.tolist(),
    })

# Output head — no BN, export directly
weights_dict = {
    # Encoder
    "encoder.conv1.weight":  w_conv1.tolist(),
    "encoder.conv1.bias":    b_conv1.tolist(),
    "encoder.conv2.weight":  w_conv2.tolist(),
    "encoder.conv2.bias":    b_conv2.tolist(),
    "encoder.proj.weight":   w_proj.tolist(),
    "encoder.proj.bias":     b_proj.tolist(),
    # Trunk
    "trunk":                 folded_blocks,
    # Output head
    "out1.weight":           model.out1.weight.data.tolist(),
    "out1.bias":             model.out1.bias.data.tolist(),
    "out2.weight":           model.out2.weight.data.tolist(),
    "out2.bias":             model.out2.bias.data.tolist(),
    # Forced-pass shortcut
    "w_forced_pass":         model.w_forced_pass.item(),
    # Precomputed band masks (for Java index convenience)
    "band1_mask":            BAND_MASK[1],
    "band2_mask":            BAND_MASK[2],
    "band3_mask":            BAND_MASK[3],
    # Training metadata
    "num_res_blocks":        NUM_RES_BLOCKS,
    "best_val_loss":         float(best_val),
    "dataset_config":        DATASET_CONFIG,
    "dataset_repo":          HF_DATASET_REPO,
}

with open(WEIGHTS_JSON, 'w') as f:
    json.dump(weights_dict, f)

import os
size_mb = os.path.getsize(WEIGHTS_JSON) / 1e6
print(f"Exported folded weights → '{WEIGHTS_JSON}' ({size_mb:.2f} MB)")

## 12. Upload to Hugging Face Hub

This cell uploads the following artefacts to `Bluefir/BandDPER`:

| File | Description |
|------|-------------|
| `banddper.pth` | Full PyTorch state dict (for resuming training) |
| `banddper_weights.json` | BN-folded weights for Java inference |
| `training_curves.png` | Loss curves |

> **Authentication:** Run `huggingface-cli login` in a terminal, or set the `HF_TOKEN` environment variable / Colab secret, before executing this cell.

In [ ]:
import os

# Retrieve token from Colab secrets (key: HF_TOKEN) or environment variable
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("HF token loaded from Colab secrets.")
except Exception:
    hf_token = os.environ.get('HF_TOKEN', None)
    if hf_token:
        print("HF token loaded from environment variable.")
    else:
        print("No HF_TOKEN found — will rely on cached `huggingface-cli login` credentials.")

api = HfApi(token=hf_token)

# Ensure the model repo exists
try:
    api.create_repo(HF_MODEL_REPO, repo_type="model", exist_ok=True)
    print(f"Repo '{HF_MODEL_REPO}' is ready.")
except Exception as e:
    print(f"Repo creation skipped or failed: {e}")

# Upload artefacts
uploads = [
    (CHECKPOINT_PTH,  CHECKPOINT_PTH,  "PyTorch state dict"),
    (WEIGHTS_JSON,    WEIGHTS_JSON,    "BN-folded weights JSON (Java inference)"),
    ("training_curves.png", "training_curves.png", "Training loss curves"),
]

repo_path = "all_40epoch_fixed"

for local_path, repo_path, desc in uploads:
    if not os.path.exists(local_path):
        print(f"  Skipping '{local_path}' (not found).")
        continue
    print(f"  Uploading {desc} → {HF_MODEL_REPO}/{repo_path} …")
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        commit_message=f"Add {desc}",
    )
    print(f"    ✓ Done.")

print(f"\nAll artefacts uploaded to: https://huggingface.co/{HF_MODEL_REPO}")